# Sistem Tanya Jawab Akademik FILKOM UB

Orie Abyan Maulana · 245150207111050  
Rionaldi Suhandafa · 245150200111045  
Affan Abyarahman · 245150207111020  
Rimba Nevada · 245150201111013  
Muhammad Faris El Hakim · 245150207111019  
Priyambodo Arundaya Satmika · 245150201111011

# 1. Ekstraksi dan Pembersihan PDF

In [1]:
from __future__ import annotations

import json
import re
from dataclasses import dataclass, field
from pathlib import Path

import pandas as pd
import pdfplumber
from pdfplumber.page import Page

BATAS_KATA = 150


TOLERANSI_X = 6.0


TOLERANSI_LANJUTAN = 1.0


TOLERANSI_Y = 5.0


def jumlah_kata(teks: str) -> int:
    return len(teks.split())


DOKUMEN = {
    "pa2020": {
        "judul": "Buku Pedoman Akademik Fakultas Ilmu Komputer Universitas Brawijaya",
        "tahun": 2020,
        "file": "Buku-Pedoman-Akademik-FILKOM-2020.pdf",
        "url": "https://filkom.ub.ac.id/wp-content/uploads/2021/09/Buku-Pedoman-Akademik-FILKOM-2020.pdf",
        "metode": "pdf",
        "halaman_mulai": 9,
        "pola_mulai": r"^VISI, MISI, DAN TUJUAN",
        "pola_buang": [],
    },
    "tatatertib": {
        "judul": "Tata Tertib Pelaksanaan Ujian (UTS & UAS) FILKOM UB",
        "tahun": 2016,
        "file": "Tata_Tertib_Pelaksanaan_UTS_dan_UAS_2.pdf",
        "url": "https://filkom.ub.ac.id/wp-content/uploads/2021/04/Tata_Tertib_Pelaksanaan_UTS_dan_UAS_2.pdf",
        "metode": "pdf",
        "halaman_mulai": 1,
        "pola_mulai": r"^PENGUMUMAN",
        "pola_buang": [r"^Demikian tata tertib"],
    },
    "penilaian": {
        "judul": "Pedoman Penilaian dan Perhitungan Indeks Prestasi PTIIK",
        "tahun": 2015,
        "file": "Pedoman_Penilaian_dan_Perhitungan_Indeks_Prestasi_PTIIK.pdf",
        "url": "https://filkom.ub.ac.id/wp-content/uploads/2021/04/Pedoman_Penilaian_dan_Perhitungan_Indeks_Prestasi_PTIIK.pdf",
        "metode": "manual",
    },
}


@dataclass
class Baris:
    teks: str
    x0: float
    hal: int


def bukan_rumus(obj: dict) -> bool:

    if obj.get("object_type") != "char":
        return True
    return "TimesNewRomanPS-Italic" not in obj.get("fontname", "") and obj["text"] != ""


def ekstrak_baris_halaman(halaman: Page, nomor: int) -> list[Baris]:
    kata = halaman.filter(bukan_rumus).extract_words()
    kata.sort(key=lambda k: (k["top"], k["x0"]))

    kelompok: list[list[dict]] = []
    for k in kata:
        if kelompok and abs(k["top"] - kelompok[-1][0]["top"]) <= TOLERANSI_Y:
            kelompok[-1].append(k)
        else:
            kelompok.append([k])

    hasil = []
    for anggota in kelompok:
        anggota.sort(key=lambda k: k["x0"])
        teks = " ".join(k["text"] for k in anggota)
        hasil.append(Baris(teks=teks, x0=round(anggota[0]["x0"], 1), hal=nomor))
    return hasil


def ekstrak_pdf(path: Path) -> list[Baris]:

    with pdfplumber.open(path) as pdf:
        return [b for i, hlm in enumerate(pdf.pages, start=1) for b in ekstrak_baris_halaman(hlm, i)]


def potong_awal(baris: list[Baris], cfg: dict) -> list[Baris]:
    kandidat = [b for b in baris if b.hal >= cfg["halaman_mulai"]]
    pola = re.compile(cfg["pola_mulai"])
    for i, b in enumerate(kandidat):
        if pola.search(b.teks):
            return kandidat[i:]
    raise ValueError(f"Pola awal {cfg['pola_mulai']!r} tidak ditemukan")


POLA_SISA_RUMUS = re.compile(r"[=+.\s\d]+")


POLA_PENANDA_SAJA = re.compile(r"\d{1,2}\.")


def buang_sampah(baris: list[Baris], cfg: dict) -> list[Baris]:
    pola_buang = [re.compile(pola) for pola in cfg["pola_buang"]]
    return [
        b for b in baris
        if b.teks != str(b.hal)
        and (not POLA_SISA_RUMUS.fullmatch(b.teks) or POLA_PENANDA_SAJA.fullmatch(b.teks))
        and not any(pola.search(b.teks) for pola in pola_buang)
    ]


PETA_SIMBOL = str.maketrans({"": "≥", "": "≤"})


PERBAIKAN_EKSTRAKSI = [
    (re.compile(r"\blPK\b"), "IPK"),
    (re.compile(r"\blP\b"), "IP"),
    (re.compile(r"\blndeks\b"), "Indeks"),
    (re.compile(r"\bIImu\b"), "Ilmu"),
    (re.compile(r"\bl \((satu)\)"), r"1 (\1)"),
]


POLA_SIMBOL_PEMBANDING = re.compile(r"\s*([≥≤])\s*")


POLA_SPASI_KATA_ULANG = re.compile(r"(?<=[a-z])- (?=[a-z])")


POLA_SPASI_SEBELUM_TANDA_BACA = re.compile(r"\s+([.,;:])")


POLA_SPASI_GANDA = re.compile(r"\s{2,}")


def normalisasi(teks: str) -> str:
    teks = teks.translate(PETA_SIMBOL)
    for pola, ganti in PERBAIKAN_EKSTRAKSI:
        teks = pola.sub(ganti, teks)
    teks = POLA_SIMBOL_PEMBANDING.sub(r" \1 ", teks)
    teks = POLA_SPASI_KATA_ULANG.sub("-", teks)
    teks = POLA_SPASI_SEBELUM_TANDA_BACA.sub(r"\1", teks)
    teks = POLA_SPASI_GANDA.sub(" ", teks)
    return teks.strip()


def potong_tabel(baris: list[Baris], daftar_tabel: list[dict]) -> list[Baris]:
    for tabel in daftar_tabel:
        mulai = next(i for i, b in enumerate(baris) if re.search(tabel["mulai"], b.teks))
        sampai = next(i for i in range(mulai + 1, len(baris)) if re.search(tabel["sampai"], baris[i].teks))
        baris = baris[:mulai] + baris[sampai:]
    return baris


def sisipkan_rumus(baris: list[Baris], daftar_rumus: list[dict]) -> list[Baris]:
    for rumus in daftar_rumus:
        i = next(i for i, b in enumerate(baris) if re.search(rumus["setelah"], b.teks))
        pengantar = baris[i]
        sisipan = Baris(rumus["teks"], x0=pengantar.x0 + 18, hal=rumus["halaman"])
        baris = baris[: i + 1] + [sisipan] + baris[i + 1:]
    return baris

Ekstraksi menggunakan posisi kata pada halaman untuk membentuk baris. Nomor halaman, sisa rumus, dan simbol hasil ekstraksi kemudian dibersihkan.

# 2. Struktur dan Passage

In [2]:
POLA_BAB = re.compile(r"^BAB\s+([IVX]+)$")


POLA_SUBBAB = re.compile(r"^([A-Z])\.\s+(.+)$")


POLA_BUTIR = re.compile(r"^(\d{1,2}|[a-z]|[ivx]{1,4})\.\s+(.*)$")


def huruf_kapital_semua(teks: str) -> bool:
    huruf = [c for c in teks if c.isalpha()]
    return len(huruf) >= 3 and all(c.isupper() for c in huruf)


def klasifikasi(teks: str) -> tuple[str, re.Match | None]:
    if m := POLA_BAB.match(teks):
        return "bab", m
    if (m := POLA_SUBBAB.match(teks)) and huruf_kapital_semua(m[2]):
        return "subbab", m
    if m := POLA_BUTIR.match(teks):
        return "butir", m
    if huruf_kapital_semua(teks) and teks[-1].isalpha():
        return "judul", None
    return "lanjutan", None


@dataclass
class Butir:
    penanda: str
    x0: float
    hal: int
    isi: list["str | Butir"] = field(default_factory=list)

    @property
    def anak(self) -> list["Butir"]:
        return [x for x in self.isi if isinstance(x, Butir)]


@dataclass
class Bagian:
    kode: str
    bab: str
    judul_bab: str
    subbab: str
    judul_subbab: str
    hal: int
    paragraf: list[tuple[int, str]] = field(default_factory=list)
    butir: list[Butir] = field(default_factory=list)


def gabung_baris(daftar: list[str]) -> str:

    teks = ""
    for baris in daftar:
        if teks.endswith("-") and baris[:1].islower():
            teks += baris
        else:
            teks = f"{teks} {baris}".strip()
    return teks


def segmen(butir: Butir) -> list[str | Butir]:

    hasil: list[str | Butir] = []
    kumpulan: list[str] = []
    for x in butir.isi:
        if isinstance(x, str):
            kumpulan.append(x)
            continue
        if kumpulan:
            hasil.append(gabung_baris(kumpulan))
            kumpulan = []
        hasil.append(x)
    if kumpulan:
        hasil.append(gabung_baris(kumpulan))
    return hasil


class ParserDokumen:
    def __init__(self) -> None:
        self.bagian: list[Bagian] = []
        self.stack: list[Butir] = []
        self.bab = ""
        self.judul_bab = ""
        self.mode_judul: str | None = None
        self.nomor_judul = 0


    def _bagian_baru(self, kode: str, subbab: str, judul: str, hal: int) -> None:
        self.bagian.append(Bagian(kode, self.bab, self.judul_bab, subbab, judul, hal))
        self.stack = []

    @property
    def aktif(self) -> Bagian:
        return self.bagian[-1]


    def _bab(self, m: re.Match, b: Baris) -> None:
        self.bab, self.judul_bab = m[1], ""
        self._bagian_baru("0", "", "", b.hal)
        self.mode_judul = "bab"

    def _subbab(self, m: re.Match, b: Baris) -> None:
        self._bagian_baru(m[1], m[1], m[2], b.hal)
        self.mode_judul = "subbab"

    def _judul(self, b: Baris) -> None:
        if self.mode_judul == "bab":
            self.judul_bab = f"{self.judul_bab} {b.teks}".strip()
            self.aktif.judul_bab = self.judul_bab
            return
        satu_kata = len(b.teks.split()) == 1
        if self.mode_judul == "subbab" and not satu_kata:
            self.aktif.judul_subbab = f"{self.aktif.judul_subbab} {b.teks}"
            return
        self.nomor_judul += 1
        self._bagian_baru(f"S{self.nomor_judul}", "", b.teks, b.hal)
        self.mode_judul = "subbab"

    def _butir(self, m: re.Match, b: Baris) -> None:
        baru = Butir(m[1], b.x0, b.hal, [m[2]])
        while self.stack and self.stack[-1].x0 > b.x0 + TOLERANSI_X:
            self.stack.pop()
        if self.stack and abs(self.stack[-1].x0 - b.x0) <= TOLERANSI_X:
            self.stack.pop()
        if self.stack:
            self.stack[-1].isi.append(baru)
        else:
            self.aktif.butir.append(baru)
        self.stack.append(baru)

    def _lanjutan(self, b: Baris) -> None:
        while self.stack and self.stack[-1].x0 > b.x0 + TOLERANSI_LANJUTAN:
            self.stack.pop()
        if self.stack:
            target = self.stack[-1]
        elif self.aktif.butir:
            target = self.aktif.butir[-1]
        else:
            self.aktif.paragraf.append((b.hal, b.teks))
            return
        target.isi.append(b.teks)


    def parse(self, baris: list[Baris]) -> list[Bagian]:
        self._bagian_baru("0", "", "", baris[0].hal)
        for b in baris:
            jenis, m = klasifikasi(b.teks)
            if jenis != "judul":
                self.mode_judul = None if jenis in ("butir", "lanjutan") else self.mode_judul
            if jenis == "bab":
                self._bab(m, b)
            elif jenis == "subbab":
                self._subbab(m, b)
            elif jenis == "judul":
                self._judul(b)
            elif jenis == "butir":
                self._butir(m, b)
            else:
                self._lanjutan(b)
        return [bg for bg in self.bagian if bg.paragraf or bg.butir]


SINGKATAN = {"No", "Prof", "Dr", "Ir", "Eng", "dll", "dsb", "dkk", "Jl", "JL", "Telp", "S", "M", "T", "Kom"}


POLA_BATAS_KALIMAT = re.compile(r"(?<=[.!?])\s+(?=[A-Z0-9“\"(])")


def pecah_kalimat(teks: str) -> list[str]:
    kalimat: list[str] = []
    for bagian_teks in POLA_BATAS_KALIMAT.split(teks):
        kata_terakhir = kalimat[-1].rstrip(".").split()[-1] if kalimat else ""
        if kalimat and kata_terakhir.split(".")[-1] in SINGKATAN:
            kalimat[-1] += " " + bagian_teks
        else:
            kalimat.append(bagian_teks)
    return kalimat


def kelompokkan_kalimat(kalimat: list[str], batas: int = BATAS_KATA) -> list[str]:
    kelompok, sekarang = [], []
    for k in kalimat:
        if sekarang and jumlah_kata(" ".join(sekarang + [k])) > batas:
            kelompok.append(" ".join(sekarang))
            sekarang = []
        sekarang.append(k)
    if sekarang:
        kelompok.append(" ".join(sekarang))
    return kelompok


def teks_sendiri(butir: Butir) -> str:

    pertama = segmen(butir)[0]
    return f"{butir.penanda}. {pertama}" if isinstance(pertama, str) else f"{butir.penanda}."


def teks_lengkap(butir: Butir) -> str:
    potongan = [f"{butir.penanda}."]
    for x in segmen(butir):
        potongan.append(teks_lengkap(x) if isinstance(x, Butir) else x)
    return " ".join(potongan)


def buat_passage(dok: str, bg: Bagian, butir_id: str, teks: str, konteks: list[str], hal: int, jenis: str) -> dict:
    kode_bab = bg.bab or "X"
    return {
        "id": f"{dok}-{kode_bab}-{bg.kode}-{butir_id}",
        "dok_id": dok,
        "halaman": hal,
        "bab": bg.bab,
        "judul_bab": bg.judul_bab,
        "subbab": bg.subbab,
        "judul_subbab": bg.judul_subbab,
        "butir": butir_id if jenis == "butir" else "",
        "konteks": " › ".join(konteks),
        "teks": teks,
        "jenis": jenis,
    }


def passage_dari_butir(dok: str, bg: Bagian, butir: Butir, jalur: list[str], konteks: list[str]) -> list[dict]:
    jalur = jalur + [butir.penanda]
    lengkap = teks_lengkap(butir)
    if jumlah_kata(lengkap) <= BATAS_KATA or not butir.anak:
        return [buat_passage(dok, bg, ".".join(jalur), lengkap, konteks, butir.hal, "butir")]


    konteks_induk = konteks + [teks_sendiri(butir)]
    isi = segmen(butir)
    if isinstance(isi[0], str):
        isi = isi[1:]
    hasil, pengantar, nomor_teks = [], None, 0
    for i, x in enumerate(isi):
        if isinstance(x, Butir):
            k = konteks_induk + ([pengantar] if pengantar else [])
            hasil += passage_dari_butir(dok, bg, x, jalur, k)
        elif any(isinstance(y, Butir) for y in isi[i + 1:]):
            pengantar = x
        else:
            nomor_teks += 1
            butir_id = ".".join(jalur + [f"t{nomor_teks}"])
            hasil.append(buat_passage(dok, bg, butir_id, x, konteks_induk, butir.hal, "butir"))
    return hasil


def passage_dari_bagian(dok: str, bg: Bagian) -> list[dict]:
    hasil, pengantar = [], []
    if bg.paragraf:
        kalimat = pecah_kalimat(gabung_baris([t for _, t in bg.paragraf]))
        if bg.butir and kalimat[-1].endswith(":"):
            pengantar = [kalimat.pop()]
        for i, kelompok in enumerate(kelompokkan_kalimat(kalimat), start=1):
            hasil.append(buat_passage(dok, bg, f"p{i}", kelompok, [], bg.paragraf[0][0], "paragraf"))
    for butir in bg.butir:
        hasil += passage_dari_butir(dok, bg, butir, [], pengantar)
    return hasil


def passage_dari_tabel(dok: str, tabel: dict, bab: str = "", judul_bab: str = "",
                       subbab: str = "", judul_subbab: str = "") -> list[dict]:
    hasil = []
    for i, baris in enumerate(tabel["baris"], start=1):
        hasil.append({
            "id": f"{dok}-{tabel['id']}-{i}",
            "dok_id": dok,
            "halaman": tabel["halaman"],
            "bab": bab,
            "judul_bab": judul_bab,
            "subbab": subbab,
            "judul_subbab": judul_subbab,
            "butir": "",
            "konteks": tabel["judul"],
            "teks": tabel["templat"].format(**baris),
            "jenis": "tabel",
        })
    return hasil

Aturan bab, butir, dan indentasi mempertahankan konteks setiap aturan. Butir yang panjang dapat dipecah menjadi beberapa passage.

# 3. Korpus

In [3]:
transkripsi = json.loads(Path("Anggota1_Korpus/data/transkripsi_manual.json").read_text(encoding="utf-8"))
dokumen_pdf = [dok for dok, cfg in DOKUMEN.items() if cfg["metode"] == "pdf"]
baris_mentah = {dok: ekstrak_pdf(Path("Anggota1_Korpus/korpus_raw") / DOKUMEN[dok]["file"]) for dok in dokumen_pdf}

hasil_parse = {}
for dok, baris in baris_mentah.items():
    isi = potong_awal(baris, DOKUMEN[dok])
    bersih = buang_sampah(isi, DOKUMEN[dok])
    normal = []
    for b in bersih:
        teks = normalisasi(b.teks)
        if teks:
            normal.append(Baris(teks, b.x0, b.hal))
    manual = transkripsi.get(dok, {})
    tanpa_tabel = potong_tabel(normal, manual.get("tabel", []))
    hasil_parse[dok] = ParserDokumen().parse(sisipkan_rumus(tanpa_tabel, manual.get("rumus", [])))

passages: list[dict] = []


for dok in dokumen_pdf:
    for bg in hasil_parse[dok]:
        passages += passage_dari_bagian(dok, bg)


for dok in dokumen_pdf:
    for tabel in transkripsi.get(dok, {}).get("tabel", []):
        bagian = next(bg for bg in hasil_parse[dok] if bg.bab == tabel["bab"] and bg.kode == tabel["subbab"])
        passages += passage_dari_tabel(
            dok, tabel, bagian.bab, bagian.judul_bab, bagian.subbab, bagian.judul_subbab
        )


for dok, cfg in DOKUMEN.items():
    if cfg["metode"] != "manual":
        continue
    manual = transkripsi[dok]
    for i, par in enumerate(manual.get("paragraf", []), start=1):
        passages.append({
            "id": f"{dok}-p{i}", "dok_id": dok, "halaman": par["halaman"],
            "bab": "", "judul_bab": "", "subbab": "", "judul_subbab": par["judul_subbab"],
            "butir": "", "konteks": "", "teks": par["teks"], "jenis": "paragraf",
        })
    for tabel in manual.get("tabel", []):
        passages += passage_dari_tabel(dok, tabel, judul_subbab=tabel["judul_subbab"])

dokumen = {
    dok: {k: cfg[k] for k in ("judul", "tahun", "url", "file", "metode")}
    | ({"catatan": transkripsi[dok]["catatan"]} if "catatan" in transkripsi.get(dok, {}) else {})
    for dok, cfg in DOKUMEN.items()
}

assert len({passage["id"] for passage in passages}) == len(passages)
pd.DataFrame(passages).groupby("dok_id").size().to_frame("jumlah_passage")

,jumlah_passage
dok_id,
pa2020,396
penilaian,18
tatatertib,26


Tabel menunjukkan jumlah passage per dokumen. ID passage diuji agar setiap bukti dapat ditelusuri ke satu lokasi sumber.

# 4. Pemrosesan Pertanyaan

In [4]:
import unicodedata

VERSI_PREPROCESS = "1.0"


STOPWORDS = frozenset("""
yang dan atau dengan adalah merupakan sebagai pada di ke oleh ini itu tersebut
untuk saya aku tolong mohon jelaskan sebutkan dimaksud maksud
apa apakah siapakah siapa kapan kapankah dimana kemana darimana
mana manakah berapa berapakah bagaimana bagaimanakah mengapa kenapa saja
""".split())


ALIASES = (
    (r"\bindeks prestasi kumulatif\b|\bip kumulatif\b", "ipk"),
    (r"\bindeks prestasi\b", "ip"),
    (r"\bsatuan kredit semester\b|\bsistem kredit semester\b", "sks"),
    (r"\bujian akhir semester\b", "uas"),
    (r"\bujian tengah semester\b", "uts"),
    (r"\bprogram studi\b", "prodi"),
)


POLA_TOKEN = re.compile(
    r"\d+(?:,\d+)?|[^\W\d_]+(?:[-'][^\W\d_]+)*(?:\+)?|[≥≤<>=%×/+*−-]",
    re.UNICODE,
)


def normalisasi_teks(teks: str) -> str:

    if not isinstance(teks, str):
        raise TypeError("Teks harus berupa str.")
    hasil = unicodedata.normalize("NFKC", teks).casefold()
    hasil = hasil.replace(">=", "≥").replace("<=", "≤")

    hasil = re.sub(r"(?<=\d)\.(?=\d)", ",", hasil)
    return " ".join(hasil.split())


def tokenisasi(teks: str) -> list[str]:

    hasil = normalisasi_teks(teks)
    for pola, pengganti in ALIASES:
        hasil = re.sub(pola, pengganti, hasil)
    return POLA_TOKEN.findall(hasil)


def preprocess(teks: str) -> list[str]:

    return [token for token in tokenisasi(teks) if token not in STOPWORDS]


def teks_index(passage: dict) -> str:

    return " ".join(
        passage.get(kolom, "")
        for kolom in ("judul_subbab", "konteks", "teks")
        if passage.get(kolom, "")
    )


def preprocess_passage(passage: dict) -> list[str]:
    return preprocess(teks_index(passage))

POLA_TANYA = re.compile(
    r"\b(?:berapa(?:kah)?|siapa(?:kah)?|kapan(?:kah)?|"
    r"(?:di|ke|dari)\s*mana|bagaimana(?:kah)?|mengapa|kenapa|"
    r"apakah|apa|mana(?:kah)?)\b"
)


def jenis_pertanyaan(teks: str) -> tuple[str, str, list[str]]:

    penanda = [m.group() for m in POLA_TANYA.finditer(teks)]
    if not penanda:
        return "TIDAK_DIKENALI", "", []
    pertama = penanda[0]
    if re.fullmatch(r"(?:di|ke|dari)\s*mana", pertama):
        jenis = "DI_MANA"
    elif pertama in {"mengapa", "kenapa"}:
        jenis = "MENGAPA"
    elif pertama == "apakah":
        jenis = "YA_TIDAK"
    else:
        jenis = pertama.removesuffix("kah").upper()
    return jenis, pertama, penanda


def tipe_jawaban(teks: str, jenis: str, penanda: str) -> tuple[str, str]:

    if jenis == "YA_TIDAK":
        return "YA_TIDAK", "pertanyaan konfirmasi"

    if jenis == "SIAPA":
        if re.search(r"\bnama(?:nya)?\b", teks):
            return "ORANG", "siapa disertai permintaan nama"
        return "JABATAN", "siapa tanpa permintaan nama; asumsi domain akademik"
    if jenis == "KAPAN":
        return "WAKTU", "kapan: tanggal, tahun, atau peristiwa awal/akhir"
    if jenis == "DI_MANA":
        return "LOKASI", "penanda lokasi"
    if jenis == "MENGAPA":
        return "ALASAN", "penanda sebab"
    if jenis in {"TIDAK_DIKENALI", "MANA"}:
        return "TIDAK_DIKENALI", "bentuk belum dicakup aturan"


    if jenis == "BERAPA":
        setelah = teks.split(penanda, 1)[1].strip()
        if re.search(r"\b(?:persen|persentase)\b|%", setelah):
            return "PERSENTASE", "berapa disertai satuan persen"
        kepala = re.sub(r"^(?:(?:jumlah|total|minimal|maksimal|minimum|maksimum)\s+)+", "", setelah)
        if re.match(r"(?:sks|satuan kredit|nilai|ipk|ip|indeks prestasi|bobot|skor|orang|kali)\b", kepala):
            return "ANGKA", "objek hitungan numerik"
        if re.match(r"(?:lama|durasi|waktu|menit|jam|hari|minggu|bulan|tahun|semester)\b", kepala):
            return "DURASI", "objek hitungan berupa lama waktu"
        return "ANGKA", "berapa: jumlah numerik sebagai default"

    if re.search(r"\b(?:rumus|menghitung|hitung|perhitungan)\b", teks):
        return "RUMUS", "permintaan rumus atau cara menghitung"
    if re.search(r"\b(?:syarat|syarat-syarat|persyaratan)\b", teks):
        return "SYARAT", "permintaan persyaratan"
    if re.search(r"\b(?:prosedur|mekanisme|langkah|cara)\b", teks) or jenis == "BAGAIMANA":
        return "PROSEDUR", "permintaan cara atau tahapan"
    if re.search(r"\b(?:apa itu|apa yang dimaksud|pengertian|definisi|arti|kepanjangan)\b", teks):
        return "DEFINISI", "permintaan makna istilah"
    if re.search(r"\b(?:persen|persentase)\b|%", teks):
        return "PERSENTASE", "permintaan persentase"
    if re.search(r"\b(?:bobot|rentang|skor|ipk|indeks prestasi kumulatif)\b", teks):
        return "ANGKA", "target besaran numerik"
    if re.search(r"\b(?:nilai huruf|huruf mutu)\b", teks):
        return "NILAI_HURUF", "target konversi ke huruf"
    if re.search(r"\b(?:durasi|lama|batas waktu)\b", teks):
        return "DURASI", "target lama waktu"
    if re.search(r"\bapa saja\b", teks):
        return "DAFTAR", "permintaan daftar"
    if jenis == "APA":
        return "DEFINISI", "apa tanpa target khusus; asumsi definisi"
    return "TIDAK_DIKENALI", "target belum dicakup aturan"


def proses_pertanyaan(pertanyaan: str) -> dict:

    normal = normalisasi_teks(pertanyaan)
    if not normal:
        raise ValueError("Pertanyaan tidak boleh kosong.")
    jenis, penanda, semua_penanda = jenis_pertanyaan(normal)
    tipe, aturan = tipe_jawaban(normal, jenis, penanda)
    tokens = preprocess(pertanyaan)
    kata_kunci = list(dict.fromkeys(tokens))
    catatan = []
    perlu_klarifikasi = tipe == "TIDAK_DIKENALI" or not kata_kunci
    if len(semua_penanda) > 1:
        perlu_klarifikasi = True
        catatan.append("Terdapat beberapa penanda tanya; pecah menjadi satu pertanyaan per masukan.")
    if jenis == "SIAPA":
        catatan.append("Korpus terutama memuat jabatan. Pemetaan SIAPA ke JABATAN adalah heuristik, bukan NER.")
    if tipe == "ORANG":
        catatan.append("Nama orang sangat terbatas; keberadaan jawaban harus diperiksa saat retrieval.")
    if tipe == "WAKTU":
        catatan.append("Kalender akademik tidak tersedia; fakta tahun atau peristiwa mungkin tersedia.")
    if tipe in {"ALASAN", "PROSEDUR", "DAFTAR", "SYARAT", "YA_TIDAK"}:
        catatan.append("Tipe ini memerlukan penanganan anggota 4 dan dapat melampaui jawaban factoid singkat.")
    if re.search(r"\b(?:terbaru|sekarang|saat ini)\b", normal):
        catatan.append("Korpus historis (FILKOM 2020, tata tertib 2016, penilaian 2015); bukan aturan terkini.")
    if perlu_klarifikasi and len(semua_penanda) <= 1:
        catatan.append("Gunakan kata tanya dan sebutkan objek akademik yang ingin ditanyakan.")
    return {
        "pertanyaan": pertanyaan,
        "normalisasi": normal,
        "token": tokenisasi(pertanyaan),
        "token_pencarian": tokens,
        "kata_kunci": kata_kunci,

        "query": " ".join(tokens),
        "jenis_pertanyaan": jenis,
        "penanda_tanya": penanda,
        "tipe_jawaban": tipe,
        "aturan_tipe": aturan,
        "status": "PERLU_KLARIFIKASI" if perlu_klarifikasi else "SIAP_RETRIEVAL",
        "catatan": catatan,
        "versi_preprocess": VERSI_PREPROCESS,
    }

with open("Anggota5_Evaluasi_Retrieval/data/pertanyaan_uji.json", encoding="utf-8") as file:
    pertanyaan = json.load(file)

passage_by_id = {p["id"]: p for p in passages}
assert len({soal["id_pertanyaan"] for soal in pertanyaan}) == len(pertanyaan)

for soal in pertanyaan:
    teks_bukti = {
        pid: f'{passage_by_id[pid]["konteks"]} {passage_by_id[pid]["teks"]}'.casefold()
        for pid in soal["passage_id"]
    }
    kutipan = soal["bukti"].casefold()
    assert any(kutipan in teks for teks in teks_bukti.values()), soal["id_pertanyaan"]
    for pid, kutipan in soal.get("bukti_per_passage", {}).items():
        assert kutipan.casefold() in teks_bukti[pid], soal["id_pertanyaan"]

hasil_pertanyaan = [
    {**soal, "query": proses_pertanyaan(soal["pertanyaan"])}
    for soal in pertanyaan
]
passage_bertoken = [
    {
        "passage_id": p["id"],
        "dok_id": p["dok_id"],
        "halaman": p["halaman"],
        "token_pencarian": preprocess_passage(p),
        "versi_preprocess": VERSI_PREPROCESS,
    }
    for p in passages
]

pd.json_normalize(hasil_pertanyaan)[
    ["id_pertanyaan", "pertanyaan", "query.tipe_jawaban", "query.token_pencarian", "query.status"]
]

,id_pertanyaan,pertanyaan,query.tipe_jawaban,query.token_pencarian,query.status
0,A2-001,Berapa sks minimal untuk menyelesaikan studi s...,ANGKA,"[sks, minimal, menyelesaikan, studi, sarjana]",SIAP_RETRIEVAL
1,A2-002,Berapa lama waktu maksimal untuk Ujian Skripsi?,DURASI,"[lama, waktu, maksimal, ujian, skripsi]",SIAP_RETRIEVAL
2,A2-003,Apa itu skripsi?,DEFINISI,[skripsi],SIAP_RETRIEVAL
3,A2-004,Apa nilai huruf untuk nilai angka 78?,NILAI_HURUF,"[nilai, huruf, nilai, angka, 78]",SIAP_RETRIEVAL
4,A3-001,Berapa lama waktu maksimal untuk menyelesaikan...,DURASI,"[lama, waktu, maksimal, menyelesaikan, studi, ...",SIAP_RETRIEVAL
5,A3-002,Kapan batas pengajuan cuti akademik setelah da...,WAKTU,"[batas, pengajuan, cuti, akademik, setelah, da...",SIAP_RETRIEVAL
6,A3-003,Apa rumus indeks prestasi semester?,RUMUS,"[rumus, ip, semester]",SIAP_RETRIEVAL
7,A4-001,Berapa persentase kehadiran minimal agar mahas...,PERSENTASE,"[persentase, kehadiran, minimal, agar, mahasis...",SIAP_RETRIEVAL
8,A4-002,Berapa menit batas maksimal keterlambatan meng...,DURASI,"[menit, batas, maksimal, keterlambatan, mengik...",SIAP_RETRIEVAL
9,A4-003,Berapa bobot nilai huruf B+?,ANGKA,"[bobot, nilai, huruf, b+]",SIAP_RETRIEVAL


Tabel menampilkan tipe jawaban, token pencarian, dan status setiap soal. Pertanyaan serta passage memakai aturan normalisasi yang sama.

# 5. Retrieval

In [5]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

token_by_id = {p["passage_id"]: p["token_pencarian"] for p in passage_bertoken}
vectorizer = TfidfVectorizer(analyzer=lambda tokens: tokens)
matriks = vectorizer.fit_transform([token_by_id[p["id"]] for p in passages])

baris_pencarian = []
for soal in hasil_pertanyaan:
    query = soal["query"]
    urutan = []
    kandidat_top5 = []
    if query["status"] == "SIAP_RETRIEVAL" and query["token_pencarian"]:
        vektor = vectorizer.transform([query["token_pencarian"]])
        skor = cosine_similarity(vektor, matriks)[0]
        urutan = sorted(
            (i for i, nilai in enumerate(skor) if nilai > 0),
            key=lambda i: (-skor[i], passages[i]["id"]),
        )
        kandidat_top5 = [
            {
                **passages[i],
                "passage_id": passages[i]["id"],
                "rank": rank,
                "score": float(skor[i]),
                "sumber": dokumen[passages[i]["dok_id"]],
            }
            for rank, i in enumerate(urutan[:5], start=1)
        ]
    baris_pencarian.append({
        **soal,
        "urutan_passage": [passages[i]["id"] for i in urutan],
        "kandidat_top5": kandidat_top5,
    })

hasil_pencarian = {"jumlah_passage": len(passages), "hasil_per_pertanyaan": baris_pencarian}

pd.DataFrame(
    {"id_pertanyaan": soal["id_pertanyaan"], "rank": p["rank"], "passage_id": p["passage_id"], "score": p["score"]}
    for soal in baris_pencarian for p in soal["kandidat_top5"]
)

,id_pertanyaan,rank,passage_id,score
0,A2-001,1,pa2020-III-J-2.1,0.264577
1,A2-001,2,pa2020-IV-A-3,0.244562
2,A2-001,3,pa2020-II-I-7,0.241728
3,A2-001,4,pa2020-VI-D-4,0.235918
4,A2-001,5,pa2020-II-F-4,0.233342
...,...,...,...,...
95,SUP-005,1,pa2020-tabel1-1,0.681192
96,SUP-005,2,pa2020-tabel1-2,0.667080
97,SUP-005,3,pa2020-tabel1-3,0.577617
98,SUP-005,4,pa2020-tabel1-4,0.576349


Tabel menunjukkan lima kandidat dan skor cosine setiap soal. Urutan semua kandidat berskor positif dipakai untuk menghitung peringkat bukti pertama.

# 6. Ekstraksi Jawaban

In [6]:
import operator

import spacy_udpipe

_UNIT_BILANGAN = (
    "sks|semester|menit|jam|hari|minggu|bulan|tahun|kali|orang|publikasi|jenis"
)


_DASAR = {
    "nol": 0, "satu": 1, "dua": 2, "tiga": 3, "empat": 4,
    "lima": 5, "enam": 6, "tujuh": 7, "delapan": 8, "sembilan": 9,
}


_KOMPARATOR = {
    "<": operator.lt,
    "≤": operator.le,
    "<=": operator.le,
    ">": operator.gt,
    "≥": operator.ge,
    ">=": operator.ge,
}


_POLA_KURUNG_ANGKA = re.compile(r"\b(\d+(?:,\d+)?)\s*\([a-z\s-]+\)", re.IGNORECASE)


_POLA_KATA_BILANGAN = re.compile(
    rf"\b(?!(?:salah\s+)?satu\b(?!\s+(?:{_UNIT_BILANGAN})\b))"
    r"(?:(?:seratus|(?:" + "|".join(_DASAR) + r")\s+ratus)\s+)?"
    r"(?:(?:sepuluh|sebelas|(?:" + "|".join(_DASAR) + r")\s+belas)|"
    r"(?:(?:" + "|".join(_DASAR) + r")\s+puluh(?:\s+(?:" + "|".join(_DASAR) + r"))?)|"
    r"(?:" + "|".join(_DASAR) + r"))"
    rf"(?=\s+(?:{_UNIT_BILANGAN})\b|\b)",
    re.IGNORECASE,
)


def _parse_frasa_bilangan(frasa: str) -> int | None:

    tokens = frasa.casefold().split()
    total = 0
    i = 0
    while i < len(tokens):
        tok = tokens[i]
        if tok == "seratus":
            total += 100
        elif tok == "sepuluh":
            total += 10
        elif tok == "sebelas":
            total += 11
        elif tok in _DASAR:
            val = _DASAR[tok]
            if i + 1 < len(tokens) and tokens[i + 1] in {"belas", "puluh", "ratus"}:
                pengali = tokens[i + 1]
                if pengali == "belas":
                    total += 10 + val
                elif pengali == "puluh":
                    total += val * 10
                else:
                    total += val * 100
                i += 1
            else:
                total += val
        else:
            return None
        i += 1
    return total


def normalisasi_bilangan(teks: str) -> str:

    hasil = _POLA_KURUNG_ANGKA.sub(r"\1", teks)

    def _ganti(match: re.Match) -> str:
        frasa = match.group(0)
        tokens = frasa.casefold().split()

        if len(tokens) == 1:
            sisa = hasil[match.end():].lstrip()
            if not re.match(rf"^(?:{_UNIT_BILANGAN})\b", sisa, re.IGNORECASE):
                return frasa
        angka = _parse_frasa_bilangan(frasa)
        return str(angka) if angka is not None else frasa

    return _POLA_KATA_BILANGAN.sub(_ganti, hasil)


def _ke_float(teks_angka: str) -> float:
    return float(teks_angka.replace(",", "."))


def _cek_rentang(teks: str, nilai: float) -> bool | None:

    dua_sisi = re.search(
        r"(\d+(?:,\d+)?)\s*([<≤]|<=)\s*[A-Za-z]+\s*([<≤]|<=)\s*(\d+(?:,\d+)?)",
        teks,
    )
    if dua_sisi:
        bawah, op1, op2, atas = dua_sisi.groups()
        return _KOMPARATOR[op1](_ke_float(bawah), nilai) and _KOMPARATOR[op2](nilai, _ke_float(atas))

    satu_sisi = re.search(r"\b[A-Za-z]+\s*([≥≤<>]|>=|<=)\s*(\d+(?:,\d+)?)", teks)
    if satu_sisi:
        op, batas = satu_sisi.groups()
        return _KOMPARATOR[op](nilai, _ke_float(batas))
    return None


def evaluasi_tabel(teks: str, pertanyaan: str, tipe_jawaban: str) -> str | None:

    angka_query = re.findall(r"\d+(?:,\d+|\.\d+)?", pertanyaan)
    if angka_query:
        nilai = _ke_float(angka_query[-1].replace(".", ","))
        cocok = _cek_rentang(teks, nilai)
        if cocok is False:
            return None
        if cocok is True:
            if tipe_jawaban == "NILAI_HURUF":
                m = re.search(r"nilai huruf\s+([A-E]\+?)", teks)
                if m:
                    return m.group(1)
            m_adalah = re.search(r"\badalah\s+([^.(]+)", teks)
            if m_adalah:
                return m_adalah.group(1).strip()


    m_huruf_q = re.findall(r"(?<![A-Za-z])([A-E]\+?)(?![A-Za-z0-9+])", pertanyaan)
    if m_huruf_q:
        huruf_target = m_huruf_q[-1]
        pola_baris = re.search(
            rf"(?:Huruf Mutu \(HM\)|nilai huruf)\s+{re.escape(huruf_target)}(?![A-Za-z0-9+])",
            teks,
        )
        if pola_baris:
            m_nilai = re.search(r"(?:Angka Mutu \(AM\)|bobot)\s+(\d+(?:,\d+)?)", teks)
            if m_nilai:
                return m_nilai.group(1)
    return None


def ekstrak_rumus(teks: str) -> str | None:

    m = re.search(r"\b([A-Z]{1,4}\s*=\s*[^.]+)", teks)
    if not m:
        return None
    rumus = re.split(r",\s*dengan\b", m.group(1), maxsplit=1)[0].strip()
    return f"{rumus}."

_UNIT_DURASI = frozenset({"menit", "jam", "hari", "minggu", "bulan", "tahun", "semester"})


_UNIT_ANGKA = frozenset({
    "sks", "ipk", "ip", "bobot", "skor", "poin", "kali", "orang", "publikasi", "jenis",
})


_BULAN_ID = frozenset({
    "januari", "februari", "maret", "april", "mei", "juni",
    "juli", "agustus", "september", "oktober", "november", "desember",
})


_NLP = None


def load_udpipe():

    global _NLP
    if _NLP is not None:
        return _NLP
    try:
        _NLP = spacy_udpipe.load("id")
    except (AssertionError, OSError):
        spacy_udpipe.download("id")
        _NLP = spacy_udpipe.load("id")
    return _NLP


def _akar_sederhana(kata: str) -> str:

    w = re.sub(r"[^a-z0-9+]", "", kata.casefold())
    if len(w) <= 4:
        return w
    if w.startswith(("meny", "peny")):
        w = "s" + w[4:]
    elif w.startswith(("men", "pen")) and len(w) > 5 and w[3] in "aeiou":
        w = "t" + w[3:]
    elif w.startswith(("mem", "pem")) and len(w) > 5 and w[3] in "aeiou":
        w = "p" + w[3:]
    elif w.startswith(("meng", "peng")) and len(w) > 6 and w[4] in "aeiou":
        w = w[4:]
    else:
        w = re.sub(r"^(?:meng|peng|mem|pem|men|pen|me|pe|di|ber|ter|ke)", "", w)
    if len(w) > 5:
        w = re.sub(r"(?:kan|an)$", "", w)
    return w


def _akar_kunci(hasil_pertanyaan: dict) -> set[str]:
    kata = hasil_pertanyaan.get("kata_kunci") or []
    return {_akar_sederhana(k) for k in kata if len(k) >= 2}


def _bersihkan_butir(teks: str) -> str:

    return re.sub(r"^\s*(?:\d+|[a-z]{1,2}|[ivx]+)\.\s+", "", teks.strip(), flags=re.IGNORECASE)


def _format_sumber(passage: dict, dokumen: dict | None = None) -> str:
    sumber = passage.get("sumber")
    if not sumber and dokumen and passage.get("dok_id") in dokumen:
        sumber = dokumen[passage["dok_id"]]
    if not sumber:
        return f"{passage.get('passage_id', '')}, hlm. {passage.get('halaman', '-')}"
    bab = passage.get("bab", "")
    subbab = passage.get("subbab", "")
    lokasi = f"BAB {bab} {subbab}".strip() if bab else passage.get("judul_subbab", "")
    bagian = f", {lokasi}" if lokasi else ""
    return f"{sumber['judul']} ({sumber['tahun']}){bagian}, hlm. {passage.get('halaman', '-')}"


def _jarak_terdekat(
    idx_token: int,
    doc,
    akar_kunci: set[str],
    kecualikan: range | None = None,
) -> int:
    cocok = [
        abs(idx_token - t.i)
        for t in doc
        if (kecualikan is None or t.i not in kecualikan)
        and _akar_sederhana(t.text) in akar_kunci
    ]
    return min(cocok) if cocok else 999


def _bonus_dependensi(token, akar_kunci: set[str]) -> float:

    node = token.head
    for langkah in range(2):
        if _akar_sederhana(node.text) in akar_kunci:
            return 1.2 if langkah == 0 else 0.6
        if node.head == node:
            break
        node = node.head
    return 0.0


def _overlap_kalimat(span_or_token, akar_kunci: set[str]) -> int:
    akar_sent = {_akar_sederhana(tok.text) for tok in span_or_token.sent}
    return len(akar_sent & akar_kunci)


def _ekstrak_kuantitatif(doc, tipe: str, akar_kunci: set[str], rank: int) -> tuple[str, float] | None:
    kandidat = []
    for t in doc:
        if t.pos_ != "NUM" or not re.search(r"\d", t.text):
            continue

        if t.i + 1 < len(doc) and doc[t.i + 1].text == ".":
            continue

        teks_num = t.text
        idx_akhir = t.i

        if teks_num.endswith(",") and t.i + 1 < len(doc) and doc[t.i + 1].pos_ == "NUM":
            teks_num = f"{teks_num}{doc[t.i + 1].text}"
            idx_akhir = t.i + 1

        teks_next = doc[idx_akhir + 1].text.casefold() if idx_akhir + 1 < len(doc) else ""
        teks_prev = doc[t.i - 1].text if t.i > 0 else ""

        if tipe == "PERSENTASE":
            if teks_next not in {"%", "persen"}:
                continue
            jawaban = f"{teks_num}%" if teks_next == "%" else f"{teks_num} {doc[idx_akhir + 1].text}"
        elif tipe == "DURASI":
            if teks_next not in _UNIT_DURASI:
                continue
            jawaban = f"{teks_num} {doc[idx_akhir + 1].text}"
        else:
            if teks_next in _UNIT_DURASI and teks_next not in akar_kunci:
                continue
            if teks_next in _UNIT_ANGKA:
                jawaban = f"{teks_num} {doc[idx_akhir + 1].text}"
            elif teks_prev in {"≥", "≤", ">", "<"}:
                jawaban = f"{teks_prev} {teks_num}"
            else:
                jawaban = teks_num

        jarak = _jarak_terdekat(t.i, doc, akar_kunci)
        overlap = _overlap_kalimat(t, akar_kunci)
        skor = (0.8 * overlap) + (2.0 / (jarak + 1.0)) + _bonus_dependensi(t, akar_kunci) + (1.0 / rank)
        kandidat.append((skor, jawaban))

    if not kandidat:
        return None
    kandidat.sort(key=lambda x: -x[0])
    return kandidat[0][1], kandidat[0][0]


def _ekstrak_definisi(doc, akar_kunci: set[str], rank: int) -> tuple[str, float] | None:
    for t in doc:
        if t.dep_ != "cop" or t.text.casefold() not in {"adalah", "merupakan", "ialah", "yaitu"}:
            continue
        subjek = [c for c in t.head.children if "subj" in c.dep_]
        teks_subjek = " ".join(tok.text for s in subjek for tok in s.subtree)
        akar_subjek = {_akar_sederhana(w) for w in teks_subjek.split()}
        overlap = len(akar_subjek & akar_kunci)
        if not overlap and akar_kunci:
            continue
        sent_end = t.sent.end
        potongan = doc[t.i + 1 : sent_end].text.strip()
        if not potongan:
            continue
        jawaban = potongan[0].upper() + potongan[1:]
        skor = overlap + (1.0 / rank)
        return jawaban, skor
    return None


def _ekstrak_jabatan(doc, akar_kunci: set[str], rank: int) -> tuple[str, float] | None:
    kandidat = []
    i = 0
    while i < len(doc):
        t = doc[i]
        if t.text[0].isupper() and t.pos_ in {"PROPN", "NOUN"} and t.dep_ != "punct":
            j = i + 1
            while j < len(doc) and doc[j].text[0].isupper() and doc[j].pos_ in {"PROPN", "NOUN"}:
                j += 1
            span = doc[i:j]
            akar_span = {_akar_sederhana(tok.text) for tok in span}

            if not akar_span.issubset(akar_kunci):
                rentang = range(i, j)
                jarak = min(_jarak_terdekat(k, doc, akar_kunci, rentang) for k in rentang)
                dep_bonus = max(_bonus_dependensi(doc[k], akar_kunci) for k in rentang)
                overlap = _overlap_kalimat(span, akar_kunci)
                skor = (0.8 * overlap) + (2.0 / (jarak + 1.0)) + (1.5 * dep_bonus) + (1.0 / rank)
                kandidat.append((skor, span.text))
            i = j
        else:
            i += 1
    if not kandidat:
        return None
    kandidat.sort(key=lambda x: -x[0])
    return kandidat[0][1], kandidat[0][0]


def _ekstrak_waktu(doc, akar_kunci: set[str], rank: int) -> tuple[str, float] | None:
    for t in doc:
        if t.pos_ != "NUM" or not re.search(r"\d", t.text):
            continue
        next_w = doc[t.i + 1].text.casefold() if t.i + 1 < len(doc) else ""
        if next_w not in _UNIT_DURASI and next_w not in _BULAN_ID:
            continue
        awal = t.i
        if awal >= 2 and doc[awal - 2 : awal].text.casefold() in {"paling lambat", "paling lama", "paling cepat"}:
            awal -= 2
        akhir = t.i + 2
        while akhir < t.sent.end and doc[akhir].text.casefold() not in {"dan", "serta", ",", ";", "."}:
            akhir += 1
        frasa = doc[awal:akhir].text.strip()
        if frasa:
            jawaban = frasa[0].upper() + frasa[1:] + "."
            skor = (2.0 / (_jarak_terdekat(t.i, doc, akar_kunci) + 1.0)) + (1.0 / rank)
            return jawaban, skor
    return None


def _kalimat_terbaik(teks: str, akar_kunci: set[str], rank: int) -> tuple[str, float] | None:
    kalimat_list = [k.strip() for k in re.split(r"(?<=[.!?])\s+", teks) if k.strip()]
    if not kalimat_list:
        return None
    terbaik = max(
        kalimat_list,
        key=lambda k: len({_akar_sederhana(w) for w in k.split()} & akar_kunci),
    )
    overlap = len({_akar_sederhana(w) for w in terbaik.split()} & akar_kunci)
    return terbaik, overlap + (1.0 / rank)


class EkstraktorJawaban:


    def __init__(self, dokumen: dict | None = None, nlp=None):
        self.dokumen = dokumen or {}
        self._nlp = nlp

    @property
    def nlp(self):
        if self._nlp is None:
            self._nlp = load_udpipe()
        return self._nlp

    def ekstrak(self, hasil_pertanyaan: dict, kandidat_passages: list[dict]) -> dict:
        pertanyaan = hasil_pertanyaan.get("pertanyaan", "")
        tipe = hasil_pertanyaan.get("tipe_jawaban", "TIDAK_DIKENALI")

        if hasil_pertanyaan.get("status") != "SIAP_RETRIEVAL" or not kandidat_passages:
            return {
                "pertanyaan": pertanyaan,
                "tipe_jawaban": tipe,
                "jawaban": "",
                "metode": "tidak_tersedia",
                "skor": 0.0,
                "passage_id": "",
                "kutipan": "",
                "sumber": "",
                "status": "TIDAK_DITEMUKAN",
            }

        akar_kunci = _akar_kunci(hasil_pertanyaan)


        for rank_idx, p in enumerate(kandidat_passages, start=1):
            rank = p.get("rank", rank_idx)
            teks = p.get("teks", "")
            if tipe == "RUMUS":
                rumus = ekstrak_rumus(teks)
                if rumus:
                    return self._bangun_hasil(pertanyaan, tipe, rumus, "simbolik", 1.0 / rank, p)
            if p.get("jenis") == "tabel" or tipe == "NILAI_HURUF":
                nilai_tabel = evaluasi_tabel(teks, pertanyaan, tipe)
                if nilai_tabel:
                    return self._bangun_hasil(pertanyaan, tipe, nilai_tabel, "simbolik", 1.0 / rank, p)


        kandidat_linguistik = []
        for rank_idx, p in enumerate(kandidat_passages, start=1):
            rank = p.get("rank", rank_idx)
            teks_bersih = normalisasi_bilangan(_bersihkan_butir(p.get("teks", "")))
            doc = self.nlp(teks_bersih)

            hasil_ling = None
            if tipe in {"ANGKA", "DURASI", "PERSENTASE"}:
                hasil_ling = _ekstrak_kuantitatif(doc, tipe, akar_kunci, rank)
            elif tipe == "DEFINISI":
                hasil_ling = _ekstrak_definisi(doc, akar_kunci, rank)
            elif tipe in {"JABATAN", "ORANG"}:
                hasil_ling = _ekstrak_jabatan(doc, akar_kunci, rank)
            elif tipe == "WAKTU":
                hasil_ling = _ekstrak_waktu(doc, akar_kunci, rank)

            if hasil_ling:
                jawaban, skor = hasil_ling
                kandidat_linguistik.append((skor, jawaban, p))

        if kandidat_linguistik:
            kandidat_linguistik.sort(key=lambda x: -x[0])
            skor_terbaik, jawaban_terbaik, passage_terbaik = kandidat_linguistik[0]
            return self._bangun_hasil(
                pertanyaan, tipe, jawaban_terbaik, "linguistik_udpipe", skor_terbaik, passage_terbaik,
            )


        top_p = kandidat_passages[0]
        fallback = _kalimat_terbaik(_bersihkan_butir(top_p.get("teks", "")), akar_kunci, top_p.get("rank", 1))
        if fallback:
            jawaban_fb, skor_fb = fallback
            return self._bangun_hasil(pertanyaan, tipe, jawaban_fb, "fallback_kalimat", skor_fb, top_p)

        return self._bangun_hasil(pertanyaan, tipe, "", "tidak_tersedia", 0.0, top_p, ditemukan=False)

    def _bangun_hasil(
        self,
        pertanyaan: str,
        tipe: str,
        jawaban: str,
        metode: str,
        skor: float,
        passage: dict,
        ditemukan: bool = True,
    ) -> dict:
        return {
            "pertanyaan": pertanyaan,
            "tipe_jawaban": tipe,
            "jawaban": jawaban,
            "metode": metode,
            "skor": round(float(skor), 4),
            "passage_id": passage.get("passage_id") or passage.get("id", ""),
            "kutipan": passage.get("teks", ""),
            "sumber": _format_sumber(passage, self.dokumen),
            "status": "DITEMUKAN" if ditemukan else "TIDAK_DITEMUKAN",
        }

ekstraktor = EkstraktorJawaban()
hasil_ekstraksi = [
    {
        "id_pertanyaan": soal["id_pertanyaan"],
        "jawaban_rujukan": soal["jawaban"],
        **ekstraktor.ekstrak(soal["query"], soal["kandidat_top5"]),
    }
    for soal in baris_pencarian
]

pd.DataFrame(hasil_ekstraksi)[
    ["id_pertanyaan", "jawaban", "jawaban_rujukan", "metode", "passage_id"]
]

,id_pertanyaan,jawaban,jawaban_rujukan,metode,passage_id
0,A2-001,144 sks,144 sks,linguistik_udpipe,pa2020-II-I-7
1,A2-002,75 menit,75 menit,linguistik_udpipe,pa2020-VI-D-10
2,A2-003,Format skripsi disusun menurut dokumen Panduan...,Karya ilmiah mahasiswa pendidikan program sarj...,fallback_kalimat,pa2020-VI-A-14
3,A2-004,"Menurut pedoman penilaian PTIIK, nilai angka (...",B+,fallback_kalimat,penilaian-konversi-1
4,A3-001,14 semester,14 semester sejak pertama kali terdaftar sebag...,linguistik_udpipe,pa2020-II-I-8
5,A3-002,Paling lambat 1 bulan setelah berakhirnya masa...,Paling lambat 1 bulan setelah masa daftar ulan...,linguistik_udpipe,pa2020-III-G-2
6,A3-003,Keberhasilan studi mahasiswa dinyatakan dengan...,IP = Σ (K_i × Na_i) / Σ K_i.,fallback_kalimat,pa2020-II-H-1
7,A4-001,80%,80%,linguistik_udpipe,pa2020-III-K-1
8,A4-002,15 menit,15 menit,linguistik_udpipe,tatatertib-X-S1-3
9,A4-003,"3,5","3,5",simbolik,pa2020-bobot-2


Kolom jawaban menunjukkan teks yang dipilih sistem. Metode dan ID passage memperlihatkan bagaimana jawaban diambil dan dari mana sumbernya.

# 7. Metrik Retrieval

In [7]:
from statistics import mean

from IPython.display import display

hasil_soal = []
for soal in hasil_pencarian["hasil_per_pertanyaan"]:
    relevan = set(soal["passage_id"])
    peringkat = next(
        (rank for rank, pid in enumerate(soal["urutan_passage"], start=1) if pid in relevan),
        None,
    )
    top5 = soal["kandidat_top5"]
    ditemukan = sorted({p["passage_id"] for p in top5} & relevan)
    hasil_soal.append({
        "id_pertanyaan": soal["id_pertanyaan"],
        "penanya": soal["penanya"],
        "sumber_kontribusi": soal["sumber_kontribusi"],
        "pertanyaan": soal["pertanyaan"],
        "jawaban_rujukan": soal["jawaban"],
        "passage_id_relevan": soal["passage_id"],
        "bukti_rujukan": soal["bukti"],
        "peringkat_bukti_pertama": peringkat,
        "passage_relevan_di_top5": ditemukan,
        "precision_at_5": len(ditemukan) / 5,
        "recall_at_5": len(ditemukan) / len(relevan),
        "kandidat_top5": top5,
    })

jumlah_hit5 = sum(bool(baris["passage_relevan_di_top5"]) for baris in hasil_soal)
peringkat_bukti = [baris["peringkat_bukti_pertama"] for baris in hasil_soal]
ringkasan_retrieval = {
    "jumlah_pertanyaan": len(hasil_soal),
    "jumlah_passage": hasil_pencarian["jumlah_passage"],
    "hit_at_5": jumlah_hit5 / len(hasil_soal),
    "hit_at_5_jumlah": jumlah_hit5,
    "mrr": mean(1 / rank if rank is not None else 0 for rank in peringkat_bukti),
    "precision_at_5_macro": mean(baris["precision_at_5"] for baris in hasil_soal),
    "recall_at_5_macro": mean(baris["recall_at_5"] for baris in hasil_soal),
    "metode": "TF-IDF unigram dan cosine similarity dari Anggota 3",
    "catatan": "Peringkat dihitung pada kandidat dengan skor positif; urutan seri memakai passage_id.",
}
hasil_retrieval = {"ringkasan": ringkasan_retrieval, "hasil_per_pertanyaan": hasil_soal}

display(pd.DataFrame([ringkasan_retrieval])[
    ["jumlah_pertanyaan", "hit_at_5_jumlah", "hit_at_5", "mrr", "precision_at_5_macro", "recall_at_5_macro"]
].round(3))
pd.DataFrame(hasil_soal)[
    ["id_pertanyaan", "pertanyaan", "peringkat_bukti_pertama", "passage_relevan_di_top5"]
]

,jumlah_pertanyaan,hit_at_5_jumlah,hit_at_5,mrr,precision_at_5_macro,recall_at_5_macro
0,20,17,0.85,0.78,0.2,0.808


,id_pertanyaan,pertanyaan,peringkat_bukti_pertama,passage_relevan_di_top5
0,A2-001,Berapa sks minimal untuk menyelesaikan studi s...,3,[pa2020-II-I-7]
1,A2-002,Berapa lama waktu maksimal untuk Ujian Skripsi?,1,[pa2020-VI-D-10]
2,A2-003,Apa itu skripsi?,19,[]
3,A2-004,Apa nilai huruf untuk nilai angka 78?,6,[]
4,A3-001,Berapa lama waktu maksimal untuk menyelesaikan...,2,[pa2020-II-I-8]
5,A3-002,Kapan batas pengajuan cuti akademik setelah da...,1,[pa2020-III-G-2]
6,A3-003,Apa rumus indeks prestasi semester?,25,[]
7,A4-001,Berapa persentase kehadiran minimal agar mahas...,1,"[pa2020-II-F-9, pa2020-III-K-1]"
8,A4-002,Berapa menit batas maksimal keterlambatan meng...,1,[tatatertib-X-S1-3]
9,A4-003,Berapa bobot nilai huruf B+?,1,"[pa2020-bobot-2, penilaian-bobot-2]"


Hit@5 menghitung soal dengan bukti relevan dalam lima besar. MRR menghitung rata-rata kebalikan peringkat bukti pertama, termasuk bukti yang muncul setelah peringkat lima.

# 8. Contoh Jawaban Bot

In [8]:
id_contoh = "FARIS-003"
peringkat_contoh = pd.DataFrame(hasil_soal).loc[:, ["id_pertanyaan", "peringkat_bukti_pertama"]]
contoh = pd.DataFrame(hasil_ekstraksi).merge(peringkat_contoh, on="id_pertanyaan", validate="one_to_one")
contoh.loc[contoh["id_pertanyaan"] == id_contoh, [
    "pertanyaan", "jawaban", "jawaban_rujukan", "peringkat_bukti_pertama", "passage_id", "sumber"
]]

,pertanyaan,jawaban,jawaban_rujukan,peringkat_bukti_pertama,passage_id,sumber
13,Berapa sks minimal yang harus dikumpulkan untu...,120 sks,138 sks,1,pa2020-VI-A-3,Buku Pedoman Akademik Fakultas Ilmu Komputer U...


Untuk FARIS-003, passage rujukan **138 SKS** berada di peringkat pertama. Sistem memilih **120 SKS** dari aturan tentang syarat membuat skripsi. Ini menunjukkan perbedaan antara menemukan bukti dan memilih jawaban.

# 9. Evaluasi Jawaban

In [9]:
from collections import Counter

AMBANG_F1 = 0.70


POLA_PENANDA_BUTIR = re.compile(r"^\s*(?:\d+(?:\.\d+)*|[a-zA-Z])[.)]\s+")


POLA_TANDA_TEPI = re.compile(r"^[\s.,;:]+|[\s.,;:]+$")


POLA_TOKEN_JAWABAN = re.compile(r"\d+(?:[,.]\d+)?|[a-z]+|%|\+|≥|≤|>|<")


def normalisasi_jawaban(teks: str) -> str:

    if not isinstance(teks, str):
        return ""
    hasil = unicodedata.normalize("NFKC", teks).casefold()
    hasil = POLA_PENANDA_BUTIR.sub("", hasil)
    hasil = hasil.replace("\u2018", "'").replace("\u2019", "'").replace("\u201c", '"').replace("\u201d", '"')
    hasil = re.sub(r"\s+", " ", hasil)
    return POLA_TANDA_TEPI.sub("", hasil)


def token_jawaban(teks: str) -> list[str]:

    return POLA_TOKEN_JAWABAN.findall(normalisasi_jawaban(teks))


def _berisi_berurutan(jarum: list[str], jerami: list[str]) -> bool:

    if not jarum or len(jarum) > len(jerami):
        return False
    for awal in range(len(jerami) - len(jarum) + 1):
        if jerami[awal : awal + len(jarum)] == jarum:
            return True
    return False


def cocok_persis(jawaban_sistem: str, jawaban_rujukan: str) -> bool:

    return bool(normalisasi_jawaban(jawaban_sistem)) and (
        normalisasi_jawaban(jawaban_sistem) == normalisasi_jawaban(jawaban_rujukan)
    )


def cocok_inti(jawaban_sistem: str, jawaban_rujukan: str) -> bool:

    token_sistem = token_jawaban(jawaban_sistem)
    token_rujukan = token_jawaban(jawaban_rujukan)
    return _berisi_berurutan(token_sistem, token_rujukan) or _berisi_berurutan(
        token_rujukan, token_sistem
    )


def f1_token(jawaban_sistem: str, jawaban_rujukan: str) -> float:

    token_sistem = token_jawaban(jawaban_sistem)
    token_rujukan = token_jawaban(jawaban_rujukan)
    if not token_sistem or not token_rujukan:
        return 0.0
    irisan = Counter(token_sistem) & Counter(token_rujukan)
    jumlah_irisan = sum(irisan.values())
    if jumlah_irisan == 0:
        return 0.0
    presisi = jumlah_irisan / len(token_sistem)
    recall = jumlah_irisan / len(token_rujukan)
    return 2 * presisi * recall / (presisi + recall)


def putusan_jawaban(jawaban_sistem: str, jawaban_rujukan: str, ambang_f1: float = AMBANG_F1) -> dict:

    skor_f1 = round(f1_token(jawaban_sistem, jawaban_rujukan), 4)
    if cocok_persis(jawaban_sistem, jawaban_rujukan):
        tingkat = "COCOK_PERSIS"
    elif cocok_inti(jawaban_sistem, jawaban_rujukan):
        tingkat = "COCOK_INTI"
    elif skor_f1 >= ambang_f1:
        tingkat = "MIRIP_PARAFRASA"
    else:
        tingkat = "TIDAK_COCOK"
    return {"tingkat": tingkat, "skor_f1": skor_f1}


def _ringkasan_per_kunci(hasil: list[dict], kunci: str) -> dict:

    rekap: dict[str, dict] = {}
    for baris in hasil:
        label = baris[kunci] or "TIDAK_DIKENALI"
        sel = rekap.setdefault(
            label,
            {"jumlah": 0, "cocok_persis": 0, "cocok_inti": 0, "mirip_parafrasa": 0, "benar_longgar": 0},
        )
        sel["jumlah"] += 1
        if baris["tingkat"] == "COCOK_PERSIS":
            sel["cocok_persis"] += 1
        if baris["tingkat"] in ("COCOK_PERSIS", "COCOK_INTI"):
            sel["cocok_inti"] += 1
        if baris["tingkat"] == "MIRIP_PARAFRASA":
            sel["mirip_parafrasa"] += 1
        if baris["tingkat"] != "TIDAK_COCOK":
            sel["benar_longgar"] += 1

    for label, sel in rekap.items():
        sel["akurasi_persis"] = round(sel["cocok_persis"] / sel["jumlah"], 4)
        sel["akurasi_inti"] = round(sel["cocok_inti"] / sel["jumlah"], 4)
        sel["akurasi_longgar"] = round(sel["benar_longgar"] / sel["jumlah"], 4)
    return dict(sorted(rekap.items()))


def ringkas(hasil: list[dict], ambang_f1: float = AMBANG_F1) -> dict:

    jumlah = len(hasil)
    hitung = Counter(baris["tingkat"] for baris in hasil)
    cocok_inti = hitung["COCOK_PERSIS"] + hitung["COCOK_INTI"]
    benar_longgar = jumlah - hitung["TIDAK_COCOK"]

    di_top5 = [baris for baris in hasil if baris["bukti_di_top5"]]
    luar_top5 = [baris for baris in hasil if not baris["bukti_di_top5"]]
    benar_inti_top5 = sum(
        1 for baris in di_top5 if baris["tingkat"] in ("COCOK_PERSIS", "COCOK_INTI")
    )
    benar_inti_luar = sum(
        1 for baris in luar_top5 if baris["tingkat"] in ("COCOK_PERSIS", "COCOK_INTI")
    )

    return {
        "jumlah_pertanyaan": jumlah,
        "akurasi_persis": round(hitung["COCOK_PERSIS"] / jumlah, 4) if jumlah else 0.0,
        "akurasi_inti": round(cocok_inti / jumlah, 4) if jumlah else 0.0,
        "akurasi_longgar": round(benar_longgar / jumlah, 4) if jumlah else 0.0,
        "jumlah_persis": hitung["COCOK_PERSIS"],
        "jumlah_cocok_inti": cocok_inti,
        "jumlah_mirip_parafrasa": hitung["MIRIP_PARAFRASA"],
        "jumlah_benar_longgar": benar_longgar,
        "jumlah_tidak_cocok": hitung["TIDAK_COCOK"],
        "f1_token_macro": round(mean(baris["skor_f1"] for baris in hasil), 4) if jumlah else 0.0,
        "ambang_f1_parafrasa": ambang_f1,
        "buktidi_top5": len(di_top5),
        "benar_inti_di_top5": benar_inti_top5,
        "akurasi_inti_di_top5": round(benar_inti_top5 / len(di_top5), 4) if di_top5 else 0.0,
        "bukti_luar_top5": len(luar_top5),
        "benar_inti_luar_top5": benar_inti_luar,
        "metode": "EkstraktorJawaban Hybrid (simbolik, linguistik UDPipe, fallback kalimat) Anggota 4",
        "catatan": "Akurasi ketat memakai kesamaan setelah normalisasi; akurasi inti menambah kecocokan sebagian; akurasi longgar menambah parafrasa dengan F1 token >= ambang.",
    }

ekstraksi_by_id = {baris["id_pertanyaan"]: baris for baris in hasil_ekstraksi}
hasil_penilaian = []
for baris in hasil_retrieval["hasil_per_pertanyaan"]:
    keluaran = ekstraksi_by_id[baris["id_pertanyaan"]]
    peringkat = baris["peringkat_bukti_pertama"]
    hasil_penilaian.append({
        "id_pertanyaan": baris["id_pertanyaan"],
        "penanya": baris["penanya"],
        "pertanyaan": baris["pertanyaan"],
        "tipe_jawaban": keluaran["tipe_jawaban"],
        "jawaban_rujukan": baris["jawaban_rujukan"],
        "jawaban_sistem": keluaran["jawaban"],
        "metode": keluaran["metode"],
        "status": keluaran["status"],
        "passage_id": keluaran["passage_id"],
        "sumber": keluaran["sumber"],
        "peringkat_bukti_pertama": peringkat,
        "bukti_di_top5": peringkat is not None and peringkat <= 5,
        **putusan_jawaban(keluaran["jawaban"], baris["jawaban_rujukan"]),
    })

ringkasan_jawaban = {
    **ringkas(hasil_penilaian),
    "per_tipe_jawaban": _ringkasan_per_kunci(hasil_penilaian, "tipe_jawaban"),
    "per_metode": _ringkasan_per_kunci(hasil_penilaian, "metode"),
}
hasil_evaluasi = {"ringkasan": ringkasan_jawaban, "hasil_per_pertanyaan": hasil_penilaian}

display(pd.DataFrame([ringkasan_jawaban])[
    ["jumlah_pertanyaan", "jumlah_persis", "jumlah_cocok_inti", "jumlah_benar_longgar",
     "akurasi_persis", "akurasi_inti", "akurasi_longgar", "f1_token_macro"]
].round(3))
pd.DataFrame(hasil_penilaian)[
    ["id_pertanyaan", "peringkat_bukti_pertama", "jawaban_sistem", "jawaban_rujukan", "tingkat"]
]

,jumlah_pertanyaan,jumlah_persis,jumlah_cocok_inti,jumlah_benar_longgar,akurasi_persis,akurasi_inti,akurasi_longgar,f1_token_macro
0,20,8,13,15,0.4,0.65,0.75,0.696


,id_pertanyaan,peringkat_bukti_pertama,jawaban_sistem,jawaban_rujukan,tingkat
0,A2-001,3,144 sks,144 sks,COCOK_PERSIS
1,A2-002,1,75 menit,75 menit,COCOK_PERSIS
2,A2-003,19,Format skripsi disusun menurut dokumen Panduan...,Karya ilmiah mahasiswa pendidikan program sarj...,TIDAK_COCOK
3,A2-004,6,"Menurut pedoman penilaian PTIIK, nilai angka (...",B+,TIDAK_COCOK
4,A3-001,2,14 semester,14 semester sejak pertama kali terdaftar sebag...,COCOK_INTI
5,A3-002,1,Paling lambat 1 bulan setelah berakhirnya masa...,Paling lambat 1 bulan setelah masa daftar ulan...,MIRIP_PARAFRASA
6,A3-003,25,Keberhasilan studi mahasiswa dinyatakan dengan...,IP = Σ (K_i × Na_i) / Σ K_i.,TIDAK_COCOK
7,A4-001,1,80%,80%,COCOK_PERSIS
8,A4-002,1,15 menit,15 menit,COCOK_PERSIS
9,A4-003,1,"3,5","3,5",COCOK_PERSIS


Tabel membandingkan jawaban aktual dengan rujukan. Akurasi inti mencakup kecocokan persis dan kecocokan token berurutan. Akurasi longgar juga menerima parafrasa yang memenuhi ambang F1.